# Лабораторная работа 2
### Выполнил: Рухлядев Алексей Павлович

Добро пожаловать в лабораторную работу!
Оформите ваше решение контеста Kaggle в соответствии с этим шаблоном. Ваша задача - показать проверяющим ход ваших рассуждений, поэтому советуем писать много комментариев к коду и приводить комментарии к логике на каждом этапе в текстовых ячейках.
Где необходимо, используйте графики для большей наглядности.

При отправке поменяйте название файла на ваши ФИО!

### EDA (исследовательский анализ данных)

В этом разделе вам необходимо провести анализ вашего датасета, интерпретировать признаки, выяснить их значимость и исследовать зависимости между ними.

In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns

my_seed = 42
random.seed(my_seed)
np.random.seed(my_seed)

In [ ]:
# Для начала просто посмотрим на данные и их формат
X = pd.read_csv('DOTA2_TRAIN_features.csv', index_col='match_id')
y = pd.read_csv('DOTA2_TRAIN_targets.csv', index_col='match_id')
data = X.join(y, on='match_id')
data["radiant_win"] = data['radiant_win'].map({True: 1, False: 0})
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
[c for c in data.columns if "r1" in c]

In [ ]:
for col in ['game_mode', 'lobby_type']:
    print(f"{col}: {data[col].unique()}")

Как и в прошлый раз, пробежимся по самим признакам.

К матчу в целом относятся следующие признаки:

1. `game_time` - время матча (момент в секундах с начала матча, когда одна из команд сломала трон другой)
2. `game_mode` - игровой режим (закодированы каким-то образом, каким именно - без понятия)
3. `lobby_type` - тип лобби (то ли публичный и приватный, то ли normal и ranked, как знать)
4. `objectives_len` - длина списка событий в матче (судя по `tome_of_knowledge`, например, ломание башен и барраков, убийство Рошана, первая кровь)
5. `chat_len` - количество сообщений в чате

Всего есть 10 игроков: 5 стороны Radiant и 5 стороны Dire. У каждого из них в датасете одинаковый набор признаков, так что рассмотрим только одного игрока. В силу большого количества признаков опишу только те, по названиям которых неочевидно, за что они отвечают.

1. `denies` - количество добитых союзных крипов (что лишает противника золота и опыта)
2. `lh` - количество добиваний (последних ударов) вражеских существ (понятно, что в основном здесь идет счет добитых крипов)
3. `stuns` - общая продолжительность всех оглушений, нанесенных игроком (в секундах?)
4. `creeps_stacked` и `camps_stacked` - количество застаканных (то есть не убивал, но уводил из зоны, чтобы появились новые) крипов и лагерей (возможно, повышает шанс на победу, так как в конечном счете обеспечивает лучший фарм опыта и золота, но это мы еще посмотрим)
5. `teamfight_participation` - процент командных боев, в которых участвовал игрок
6. `obs_placed` - количество поставленных наблюдательных вардов (observer ward) (Цитата с wiki: Observer Ward важен для достижения победы, так как позволяет вам наблюдать за нужными областями и шпионить за врагами, обеспечивая безопасность вам и вашим союзникам. Очень важно ставить варды в ключевых местах и уничтожать вражеские варды.)
7. `sen_placed` - количество поставленных Sentry Ward (Несколько советов с wiki, которые опять говорят о важности для победы: Sentry Ward лучше использовать в схватке с героями, которые часто полагаются на невидимость (или используют невидимость для побега). Sentry Ward установленный рядом с башней, отлично помогает против ганкающих невидимых героев или помогает совершить неожиданный ганк на невидимого врага. У Sentry Ward большой радиус обнаружения, что делает его полезным для наблюдения за невидимыми героями в ключевых местах, если у вас есть обзор над этой областью.
В основном Sentry Ward используют для обнаружения вражеских вардов, так как вражеские варды способны обнаружить варды вашей команды.).

In [ ]:
# Посмотрим на корреляции между признаками и таргетом (ну и для удобного отображения по значимости возьмем модуль)
corr_coefs = {}
coefs_data = pd.DataFrame(columns=['feature_name', 'correlation'])
for col in data.columns:
    if col == 'radiant_win':
        continue
    corr_coefs[col] = np.abs(np.corrcoef(data[col], data['radiant_win'])[0, 1])


coefs_data['feature_name'] = corr_coefs.keys()
coefs_data['correlation'] = corr_coefs.values()

coefs_data = coefs_data.sort_values(by='correlation', ascending=False)

In [ ]:
coefs_data.to_dict()

In [ ]:
coefs_data.loc[list(range(5))]

In [ ]:
coefs_data[coefs_data["feature_name"].str.contains("firstblood_claimed")]

Забавно, наибольшее значение имеет то, в каком углу столпились игроки в конце игры: правый верхний - вероятно, ломают трон Dire, левый нижний - трон Radiant. Наверное, полезно будет ввести какую-нибудь переменную для отслеживания угла с большинством игроков.

Также для победы важно как можно меньше умирать, ломать как можно больше башен, делать много убийств и ассистов, эффективно использовать руны.

Ладно, мои предположения о важности вардов и стаков оказались ошибочны, они находятся внизу по значимости (хотя это и не отрицает, что в совокупности с чем-то это важно). Ожидаемо низко по значимости также находятся признаки, относящиеся к самому матчу, а не игрокам (кроме `objectives_len` и `chat_len`: одно, наверное, важно из-за усиления крипов, использование aegis'а и прочего, второе - из-за более эффективной командной работы?).

Вместе с этим видим, что в убийствах Radiant есть пропуски - впоследлствии заполним их из `tome_of_knowledge`

### Preprocessing (подготовка данных)

 В этом разделе вам необходимо реализовать подготовку ваших данных, в том числе заполнение пропусков, фильтрацию выбросов, кодирование категориальных признаков и т.д. В этот же раздел включите любые операции над данными, которые сочтете нужными.

In [ ]:
# Еще раз убедимся, что пропуски есть только в убийствах сил света
for col in data.columns:
    if data[col].isna().sum() != 0:
        print(f'{col}: {data[col].isna().sum()} missing values')

In [ ]:
# Поазгружаем полезную инфу из tome_of_knowledge
json_path = "./tome_of_knowledge.jsonl"

hero_stats = ["kills", "deaths", "assists", "denies", "gold", "lh", "xp", "health", "max_health", "max_mana", "level", "x", "y", "stuns", "creeps_stacked", "camps_stacked", "rune_pickups", "firstblood_claimed", "teamfight_participation", "towers_killed", "roshans_killed", "obs_placed", "sen_placed"]
named_hero_stats = []
for t in [[f"r{i}_{stat}" for stat in hero_stats] for i in range(1, 6)] + [[f"d{i}_{stat}" for stat in hero_stats] for i in range(1, 6)]:
    named_hero_stats += t

json_data = pd.DataFrame(columns=["match_id"] + named_hero_stats)
json_data = json_data.set_index('match_id')

lines = pd.read_json(json_path, lines=True, chunksize=100)
for chunk in lines:
    for match_id, players in chunk[["match_id", "players"]].values:
        line_dict = dict()
        
        # for obj in objectives:
        #     if obj["type"] == "CHAT_MESSAGE_FIRSTBLOOD":
        #         line_dict["fb_time"] = obj["time"]
        #         line_dict["r_fb"] = int(obj["slot"] < 5)
        #         break
        
        for i in range(10):
            for stat in hero_stats:
                if i < 5:
                    line_dict[f"r{i+1}_{stat}"] = players[i][stat]
                else:
                    line_dict[f"d{i-4}_{stat}"] = players[i][stat]
        
        json_data.loc[match_id] = line_dict

json_data.to_csv("json_data.csv")

In [ ]:
json_data = pd.read_csv('json_data.csv', index_col="match_id")

In [ ]:
# Наконец заполним пропуски (и заменим все случайные ошибки на значения из tome_of_knowledge)
def filler(df: pd.DataFrame):
    for col in json_data.columns:
        df[col] = json_data.loc[df.index][col]

In [ ]:
# Почистим выбросы
def untrash(df: pd.DataFrame):
    for col in df.columns:
        if col in ['game_mode', 'lobby_type', 'objectives_len', 'chat_len', 'radiant_win'] or col.count('hero_id') or col.count('kills') or col.count('deaths') or col.count('denies') or col.count('firstblood_claimed'):
            continue
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # некоторые признаки имеют много нулевых значений, из-за чего другие данные в них будут ложно классифироваться как выбросы
        if upper_bound > 0:
            df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
            df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

In [ ]:
# Время feature-engineering'а
def add_teams_stats(df: pd.DataFrame):
    new_columns = {
        'r_gold': df[[f"r{i}_gold" for i in range(1, 6)]].sum(axis=1),
        'r_xp': df[[f"r{i}_xp" for i in range(1, 6)]].sum(axis=1),
        'r_kills': df[[f"r{i}_kills" for i in range(1, 6)]].sum(axis=1),
        'r_tower_killed': df[[f"r{i}_towers_killed" for i in range(1, 6)]].sum(axis=1),
        'd_gold': df[[f"d{i}_gold" for i in range(1, 6)]].sum(axis=1),
        'd_xp': df[[f"d{i}_xp" for i in range(1, 6)]].sum(axis=1),
        'd_kills': df[[f"d{i}_kills" for i in range(1, 6)]].sum(axis=1),
        'd_tower_killed': df[[f"d{i}_towers_killed" for i in range(1, 6)]].sum(axis=1),
        'r_players_top_right': sum((df[f"r{i}_x"] > 127) & (df[f"r{i}_y"] > 127) for i in range(1, 6)),
        'd_players_bottom_left': sum((df[f"d{i}_x"] < 127) & (df[f"d{i}_y"] < 127) for i in range(1, 6))
    }
    
    return pd.concat([df, pd.DataFrame(new_columns)], axis=1)

def add_both_teams_stats(df: pd.DataFrame):
    new_columns = {
        'team_gold_diff': df["r_gold"] - df["d_gold"],
        'team_xp_diff': df["r_xp"] - df["d_xp"],
        'team_kills_diff': df["r_kills"] - df["d_kills"],
        'team_towers_diff': df["r_tower_killed"] - df["d_tower_killed"]
    }
    # df["most_players_top_right"] = (sum((df[f"r{i}_x"] > 127) & (df[f"r{i}_y"] > 127) for i in range(1, 6)) + sum((df[f"d{i}_x"] > 127) & (df[f"d{i}_y"] > 127) for i in range(1, 6))) > 5

    return pd.concat([df, pd.DataFrame(new_columns)], axis=1)

In [ ]:
filler(data)
untrash(data)
data = add_teams_stats(data)
data = add_both_teams_stats(data)

In [ ]:
# Еще раз посмотрим на корреляции и отбросим совсем малозначимые признаки
corr_coefs = {}
coefs_data = pd.DataFrame(columns=['feature_name', 'correlation'])
for col in data.columns:
    if col == 'radiant_win':
        continue
    corr_coefs[col] = np.abs(np.corrcoef(data[col], data['radiant_win'])[0, 1])


coefs_data['feature_name'] = corr_coefs.keys()
coefs_data['correlation'] = corr_coefs.values()

coefs_data = coefs_data.sort_values(by='correlation', ascending=False)
features = coefs_data[coefs_data['correlation'] > 0.005]['feature_name']
features

In [ ]:
X = data[features].copy()
y = data["radiant_win"].copy()

In [ ]:
test = pd.read_csv("DOTA2_TEST_features.csv", index_col="match_id")
filler(test)
untrash(test)
test = add_teams_stats(test)
test = add_both_teams_stats(test)
test = test[features].copy()

### Model & training (Выбор модели и её обучение)

В этом разделе описываете модель и ставите эксперименты по обучению.

Если вы ставили много экспериментов, приведите их в хронологическом порядке чтобы мы увидели эволюцию ваших рассуждений.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, KFold
from catboost import CatBoostClassifier

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=my_seed)

#### Эксперимент 1

In [ ]:
%%time

# Для начала попробуем случайный лес (простенькая модель, которая даст какой-то начальный скор, от которого будем дальше отталкиваться)
rf = RandomForestClassifier(n_estimators=400, max_depth=20, min_samples_leaf=3, random_state=my_seed)
scores = cross_val_score(rf, X, y, scoring="roc_auc", cv=cv, n_jobs=-1)
scores.mean()

In [ ]:
%%time

rf.fit(X, y)

In [ ]:
y_test = rf.predict_proba(test)
submission = pd.DataFrame(y_test[:, 1], index=test.index, columns=['radiant_win'])
submission.to_csv('submission_rf.csv')

#### Эксперимент 2

In [ ]:
%%time

# Попробуем обычный градиентный бустинг
grad = GradientBoostingClassifier(random_state=my_seed)
scores = cross_val_score(grad, X, y, scoring="roc_auc", cv=cv, n_jobs=-1)
scores.mean()

In [ ]:
%%time

grad.fit(X, y)

In [ ]:
y_test = grad.predict_proba(test)
submission = pd.DataFrame(y_test[:, 1], index=test.index, columns=['radiant_win'])
submission.to_csv('submission_grad.csv')

#### Эксперимент 3

In [ ]:
old_X = X.copy()

In [ ]:
X = old_X.copy()

In [ ]:
# Попробуем выбросить некоторые из фич
X.drop(['r_gold', 'd_gold', 'r_kills', 'd_kills', 'r_xp', 'd_xp', 'game_mode', 'chat_len', 'objectives_len'], axis=1, inplace=True)

In [ ]:
%%time

grad = GradientBoostingClassifier(random_state=my_seed)
scores = cross_val_score(grad, X, y, scoring="roc_auc", cv=cv, n_jobs=-1)
scores.mean()

In [ ]:
%%time

grad.fit(X, y)

In [ ]:
old_test = test.copy()

In [ ]:
test = old_test.copy()

In [ ]:
test.drop(['r_gold', 'd_gold', 'r_kills', 'd_kills', 'r_xp', 'd_xp', 'game_mode', 'chat_len', 'objectives_len'], axis=1, inplace=True)

In [ ]:
y_test = grad.predict_proba(test)
submission = pd.DataFrame(y_test[:, 1], index=test.index, columns=['radiant_win'])
submission.to_csv('submission_grad_2.csv')

Ладно, это не помогло

#### Эксперимент 4

In [ ]:
%%time

# Попробуем CatBoost

cat = CatBoostClassifier(random_state=my_seed, verbose=1, cat_features=["r_players_top_right", "d_players_bottom_left"])
scores = cross_val_score(cat, X, y, scoring="roc_auc")
scores.mean()

In [ ]:
%%time

cat.fit(X, y)

In [ ]:
imp_df = cat.get_feature_importance(prettified=True)

In [ ]:
imp_df

In [ ]:
y_test = cat.predict_proba(test)
submission = pd.DataFrame(y_test[:, 1], index=test.index, columns=['radiant_win'])
submission.to_csv('submission_cat.csv')

#### Эксперимент 5

In [ ]:
%%time

# Попробуем подобрать оптимальные параметры для улучшения оценки

cat_features = ["r_players_top_right", "d_players_bottom_left"]
grid = CatBoostClassifier(random_seed=my_seed, eval_metric="AUC", cat_features=cat_features, verbose=True)
params = {
    'iterations': range(100, 1000, 50),
    'depth': [3, 5, 7],
    "leaf_estimation_iterations": [10, 20],
}
grid_res = grid.grid_search(params, X, y, cv=3)

In [ ]:
grid_res

In [ ]:
%%time

best_grid = CatBoostClassifier(iterations=932, depth=3, leaf_estimation_iterations=20, random_seed=my_seed, verbose=True)
scores = cross_val_score(best_grid, X, y, scoring="roc_auc")
scores.mean()

In [ ]:
%%time

best_grid.fit(X, y)

In [ ]:
y_test = best_grid.predict_proba(test)
submission = pd.DataFrame(y_test[:, 1], index=test.index, columns=['radiant_win'])
submission.to_csv('submission_grid.csv')

Что ж, это хуже чем со стандартными параметрами. Видимо, я не нашел тот гиперпараметр, который нужно подбирать

### Evaluation (оценка качества модели)

В этом разделе проводите оценку качества вашей итоговой модели.

In [ ]:
model = cat
y_pred = model.predict_proba(X)[:, 1]

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc

roc_auc = roc_auc_score(y, y_pred)
print(f'ROC-AUC: {roc_auc}')

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y, y_pred)
roc_auc_lr = auc(fpr_lr, tpr_lr)

plt.figure(figsize=(5, 4))
plt.plot(fpr_lr, tpr_lr, label=f"AUC = {roc_auc_lr:.4f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.title('ROC-кривая')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

In [ ]:
plt.hist(y.values.astype(int), histtype='step', label='true_val')
plt.hist(model.predict(X).astype(int), label='predict_val')
plt.legend()
plt.show()

### Conclusion (Выводы)

В этом разделе описываете полученные результаты и проводите анализ выполненной работы.
Что получилось / не получилось и почему?

Получилось получить данные из большого jsonl-файла. Вероятно, из них можно получить куда более интересные фичи, но я не знаток доты.

Попробовал несколько моделей. Все они получились довольно близки по скору. Лучший результат дал CatBoost. При этом подбор гиперпараметров дал худший результат относительно стандартных. Видимо, я не нащупал нужное сочетание или просто выбирал не те параметры для прогона поиска.

Среди введенных признаков неожиданно выстрелили разницы команд по убийствам, золоту и опыту и ожидаемо - количество игроков у трона. Как мне кажется, есть еще такие же важные сочетания, о которых я даже не догадываюсь.

Также в использованных моделях оказалось удобно, что в отличие от линейных моделей здесь нет надобности скейлить данные.

Как и в прошлый раз, было забавно погружаться в незнакомую сферу для изучения данных.